# Experiment 6: Cross-Model Generalization

Experiments 2-5 used a single dense embedding model (`all-MiniLM-L6-v2`). This experiment re-runs the full pipeline
(Dense, Hybrid, Rerank, Oracle, Rule-Router, Learned-Router) with a second, architecturally different embedding model
(`all-mpnet-base-v2`) on the **same 60 docs and 180 questions**.

Upload `docs.jsonl` and `qa_final_180.jsonl` when prompted.

In [1]:
!pip install -q sentence-transformers rank_bm25 scipy scikit-learn

## Step 1: Upload data

In [9]:
from google.colab import drive
import os, json, re
import numpy as np
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Experiment 6 data folder
DATA_DIR = '/content/drive/MyDrive/AI_Project/RAG Research/Exp6/data'

# Load files directly from Google Drive
docs = [
    json.loads(l)
    for l in open(os.path.join(DATA_DIR, 'docs.jsonl'), encoding='utf-8')
]

qa = [
    json.loads(l)
    for l in open(os.path.join(DATA_DIR, 'qa_final_180.jsonl'), encoding='utf-8')
]

texts = [d['text'] for d in docs]
ids = [d['id'] for d in docs]

targets = [item['expected_doc_id'] for item in qa]

true_categories = [item['category'] for item in qa]

n = len(qa)

print(f"{len(docs)} docs, {n} questions")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
60 docs, 180 questions


## Step 2: Pipeline as a function of the embedding model

In [3]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

tokenized_corpus = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)  # embedding-independent
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  # fixed across models
doc_text_by_id = {d['id']: d['text'] for d in docs}

def _norm(x):
    rng = x.max() - x.min()
    return (x - x.min()) / (rng + 1e-9)

def hit_at_k(ranked, target, k): return int(target in ranked[:k])
def rr(ranked, target): return 1.0/(ranked.index(target)+1) if target in ranked else 0.0

COST = {"dense": 1.0, "hybrid": 1.1, "hybrid_rerank": 1.1 + 10*2.0}
COST_ORDER = ["dense", "hybrid", "hybrid_rerank"]  # cheapest first, used for oracle tie-breaking

def run_full_pipeline(model_name, alpha=0.75):
    model = SentenceTransformer(model_name)
    doc_embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)

    def dense_scores(query):
        q_emb = model.encode([query], show_progress_bar=False)
        return -np.linalg.norm(doc_embeddings - q_emb, axis=1)

    def rank_dense(query, k):
        s = dense_scores(query)
        return [ids[i] for i in np.argsort(s)[::-1][:k]]

    def rank_hybrid(query, k):
        bm25_scores = np.array(bm25.get_scores(query.lower().split()))
        combined = alpha*_norm(dense_scores(query)) + (1-alpha)*_norm(bm25_scores)
        return [ids[i] for i in np.argsort(combined)[::-1][:k]]

    def rank_hybrid_rerank(query, k, shortlist_size=10):
        shortlist = rank_hybrid(query, shortlist_size)
        scores = reranker.predict([(query, doc_text_by_id[d]) for d in shortlist])
        return [shortlist[i] for i in np.argsort(scores)[::-1]][:k]

    METHODS = {"dense": rank_dense, "hybrid": rank_hybrid, "hybrid_rerank": rank_hybrid_rerank}
    results_by_method = {m: [] for m in METHODS}
    for item in qa:
        for name, fn in METHODS.items():
            results_by_method[name].append(fn(item['question'], 5))

    # Oracle: best RR, ties broken toward the CHEAPEST method; also flag queries where methods actually differ
    oracle_label, informative = [], []
    for i in range(n):
        rrs = {m: rr(results_by_method[m][i], targets[i]) for m in COST_ORDER}
        best = max(rrs.values())
        oracle_label.append(next(m for m in COST_ORDER if rrs[m] == best))
        informative.append(len(set(rrs.values())) > 1)

    def eval_routing(choices):
        h1,h3,h5,rrs,costs = [],[],[],[],[]
        for i,m in enumerate(choices):
            r = results_by_method[m][i]; t = targets[i]
            h1.append(hit_at_k(r,t,1)); h3.append(hit_at_k(r,t,3)); h5.append(hit_at_k(r,t,5))
            rrs.append(rr(r,t)); costs.append(COST[m])
        return {"recall@1": np.mean(h1), "recall@3": np.mean(h3), "recall@5": np.mean(h5),
                "mrr": np.mean(rrs), "avg_cost": np.mean(costs), "_h1": np.array(h1)}

    def featurize(query):
        toks = query.lower().split()
        has_code = float(bool(re.search(r'\b[45]\d\d\b', query)))
        has_us = float(bool(re.search(r'[a-z]+_[a-z]+', query.lower())))
        bm25_n = _norm(np.array(bm25.get_scores(toks)))
        dense_n = _norm(dense_scores(query))
        return {"n_tokens": len(toks), "has_status_code": has_code, "has_underscore_term": has_us,
                "bm25_top": float(bm25_n.max()), "dense_top": float(dense_n.max()),
                "score_gap": float(dense_n.max()-bm25_n.max())}

    X = pd.DataFrame([featurize(item['question']) for item in qa])
    y_category = np.array(true_categories)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_cat = np.empty(n, dtype=object)
    for tr, te in skf.split(X, y_category):
        sc = StandardScaler().fit(X.iloc[tr])
        clf = LogisticRegression(max_iter=1000).fit(sc.transform(X.iloc[tr]), y_category[tr])
        oof_cat[te] = clf.predict(sc.transform(X.iloc[te]))

    ROUTE_METHOD = {'keyword':'dense','troubleshooting':'hybrid','semantic':'hybrid_rerank'}
    cat_router_choices = [ROUTE_METHOD[c] for c in oof_cat]

    def rule_predict(query):
        toks = query.lower().split()
        has_code = bool(re.search(r'\b[45]\d\d\b', query)) or bool(re.search(r'[a-z]+_[a-z]+', query.lower()))
        if has_code and len(toks)<12: return 'keyword'
        if len(toks)>=14: return 'troubleshooting'
        return 'semantic'
    rule_choices = [ROUTE_METHOD[rule_predict(item['question'])] for item in qa]

    summary = {
        "Always dense": eval_routing(['dense']*n),
        "Always hybrid": eval_routing(['hybrid']*n),
        "Always hybrid_rerank": eval_routing(['hybrid_rerank']*n),
        "Rule router": eval_routing(rule_choices),
        "Learned category router (CV)": eval_routing(cat_router_choices),
        "Oracle": eval_routing(oracle_label),
    }
    return {"summary": summary, "oracle_label": oracle_label, "informative": informative,
            "category_cv_accuracy": float((oof_cat==y_category).mean())}

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Step 3: Run both models

**Sanity check first:** Model A (MiniLM) numbers must match Experiments 2-5 (e.g. learned router ~78.3%). If they don't, debug before trusting Model B.
Alpha is fixed at 0.75 for both models (it was tuned on MiniLM) - mention this in the paper.

In [4]:
print("=== Model A: all-MiniLM-L6-v2 ===")
result_A = run_full_pipeline('sentence-transformers/all-MiniLM-L6-v2')
print("=== Model B: all-mpnet-base-v2 ===")
result_B = run_full_pipeline('sentence-transformers/all-mpnet-base-v2')

print("\nSANITY CHECK (Model A vs Exp 2-5):")
for k, v in result_A['summary'].items():
    print(f"  {k:32s} R@1={v['recall@1']:.3f}  MRR={v['mrr']:.3f}")

=== Model A: all-MiniLM-L6-v2 ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


SANITY CHECK (Model A vs Exp 2-5):
  Always dense                     R@1=0.722  MRR=0.801
  Always hybrid                    R@1=0.761  MRR=0.831
  Always hybrid_rerank             R@1=0.717  MRR=0.800
  Rule router                      R@1=0.761  MRR=0.832
  Learned category router (CV)     R@1=0.783  MRR=0.849
  Oracle                           R@1=0.867  MRR=0.907


## Step 4: Side-by-side table

In [5]:
rows = []
for label, res in [("MiniLM-L6-v2", result_A), ("mpnet-base-v2", result_B)]:
    for name, r in res['summary'].items():
        rows.append({"model": label, "method": name, "recall@1": r['recall@1'], "recall@3": r['recall@3'],
                     "recall@5": r['recall@5'], "mrr": r['mrr'], "avg_cost": r['avg_cost']})
cross_model_df = pd.DataFrame(rows)
cross_model_df.round(3)

,model,method,recall@1,recall@3,recall@5,mrr,avg_cost
0,MiniLM-L6-v2,Always dense,0.722,0.856,0.928,0.801,1.000
1,MiniLM-L6-v2,Always hybrid,0.761,0.911,0.933,0.831,1.100
2,MiniLM-L6-v2,Always hybrid_rerank,0.717,0.872,0.922,0.800,21.100
3,MiniLM-L6-v2,Rule router,0.761,0.911,0.933,0.832,4.643
4,MiniLM-L6-v2,Learned category router (CV),0.783,0.917,0.961,0.849,6.953
5,MiniLM-L6-v2,Oracle,0.867,0.939,0.972,0.907,3.355
6,mpnet-base-v2,Always dense,0.772,0.889,0.911,0.832,1.000
7,mpnet-base-v2,Always hybrid,0.767,0.906,0.933,0.835,1.100
8,mpnet-base-v2,Always hybrid_rerank,0.706,0.872,0.922,0.793,21.100
9,mpnet-base-v2,Rule router,0.767,0.906,0.933,0.835,4.643


## Step 5: Do the qualitative claims hold per model?
With paired bootstrap significance for Learned Router vs best fixed strategy (earlier experiments found this gap was not significant, so frame results directionally unless the CI excludes 0).

In [6]:
def paired_bootstrap(a, b, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed); nq = len(a)
    idxs = rng.integers(0, nq, size=(n_boot, nq))
    diffs = a[idxs].mean(axis=1) - b[idxs].mean(axis=1)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"observed_diff": float(a.mean()-b.mean()), "ci_95": (float(lo), float(hi)),
            "p_one_sided": float((diffs <= 0).mean())}

for label, r in [("MiniLM-L6-v2", result_A), ("mpnet-base-v2", result_B)]:
    s = r['summary']
    best_fixed = max(['Always dense','Always hybrid','Always hybrid_rerank'], key=lambda k: s[k]['recall@1'])
    lr = s['Learned category router (CV)']
    print(f"--- {label} ---")
    print(f"  Hybrid beats Dense? {s['Always hybrid']['recall@1'] > s['Always dense']['recall@1']} "
          f"({s['Always hybrid']['recall@1']:.3f} vs {s['Always dense']['recall@1']:.3f})")
    print(f"  Best fixed: {best_fixed} ({s[best_fixed]['recall@1']:.3f})")
    print(f"  Learned router vs best fixed: {lr['recall@1']:.3f} vs {s[best_fixed]['recall@1']:.3f}")
    print(f"  Bootstrap: {paired_bootstrap(lr['_h1'], s[best_fixed]['_h1'])}")
    print(f"  Oracle gap (Oracle - best fixed): {s['Oracle']['recall@1'] - s[best_fixed]['recall@1']:.3f}")
    print(f"  Category-classifier CV accuracy: {r['category_cv_accuracy']:.3f}\n")

--- MiniLM-L6-v2 ---
  Hybrid beats Dense? True (0.761 vs 0.722)
  Best fixed: Always hybrid (0.761)
  Learned router vs best fixed: 0.783 vs 0.761
  Bootstrap: {'observed_diff': 0.022222222222222254, 'ci_95': (-0.01666666666666672, 0.06666666666666665), 'p_one_sided': 0.1696}
  Oracle gap (Oracle - best fixed): 0.106
  Category-classifier CV accuracy: 0.939

--- mpnet-base-v2 ---
  Hybrid beats Dense? False (0.767 vs 0.772)
  Best fixed: Always dense (0.772)
  Learned router vs best fixed: 0.767 vs 0.772
  Bootstrap: {'observed_diff': -0.005555555555555536, 'ci_95': (-0.04444444444444451, 0.033333333333333326), 'p_one_sided': 0.6694}
  Oracle gap (Oracle - best fixed): 0.083
  Category-classifier CV accuracy: 0.939



## Step 6: Oracle agreement between models
Ties are broken toward the cheapest method, and agreement is reported both overall and only on queries where methods actually differ (in at least one of the models), to avoid inflating agreement with trivial ties.

In [7]:
oA, oB = result_A['oracle_label'], result_B['oracle_label']
agree_all = np.mean([a==b for a,b in zip(oA,oB)])
mask = [ia or ib for ia,ib in zip(result_A['informative'], result_B['informative'])]
agree_inf = np.mean([a==b for a,b,m in zip(oA,oB,mask) if m]) if any(mask) else float('nan')
print(f"Agreement (all queries): {agree_all:.3f}")
print(f"Agreement (informative queries only, n={sum(mask)}): {agree_inf:.3f}")

a_h1 = result_A['summary']['Always hybrid']['_h1']; b_h1 = result_B['summary']['Always hybrid']['_h1']
print("Always-Hybrid R@1, MiniLM vs mpnet:", paired_bootstrap(a_h1, b_h1))

Agreement (all queries): 0.822
Agreement (informative queries only, n=73): 0.562
Always-Hybrid R@1, MiniLM vs mpnet: {'observed_diff': -0.005555555555555647, 'ci_95': (-0.061111111111111116, 0.04444444444444451), 'p_one_sided': 0.6342}


## Step 7: Save

In [10]:
def clean(s):
    return {k: v for k, v in s.items() if k != '_h1'}


export = {

    "model_A": {
        "name": "all-MiniLM-L6-v2",
        "summary": {
            k: clean(v)
            for k, v in result_A['summary'].items()
        },
        "category_cv_accuracy": result_A['category_cv_accuracy']
    },

    "model_B": {
        "name": "all-mpnet-base-v2",
        "summary": {
            k: clean(v)
            for k, v in result_B['summary'].items()
        },
        "category_cv_accuracy": result_B['category_cv_accuracy']
    },

    "oracle_agreement_all": float(agree_all),

    "oracle_agreement_informative": float(agree_inf),

    "n_informative_queries": int(sum(mask)),
}


# Save Experiment 6 results directly to Google Drive
with open(
    os.path.join(DATA_DIR, 'experiment6_results.json'),
    'w',
    encoding='utf-8'
) as f:
    json.dump(export, f, indent=2, default=float)


cross_model_df.to_csv(
    os.path.join(DATA_DIR, 'experiment6_cross_model_table.csv'),
    index=False
)


print("Experiment 6 files saved to Google Drive:")
print(os.path.join(DATA_DIR, 'experiment6_results.json'))
print(os.path.join(DATA_DIR, 'experiment6_cross_model_table.csv'))

Experiment 6 files saved to Google Drive:
/content/drive/MyDrive/AI_Project/RAG Research/Exp6/data/experiment6_results.json
/content/drive/MyDrive/AI_Project/RAG Research/Exp6/data/experiment6_cross_model_table.csv


## Paper notes
- Frame claims as "holds across the two models tested", not "universally".
- Report alpha=0.75 fixed for both models (tuned on MiniLM).
- Keep the router claim directional unless the bootstrap CI excludes 0.
- Then update the Limitations bullet on single-embedding-model.